In [1]:
!nvidia-smi

Thu Nov 19 06:14:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install -q "transformers>=4.57"

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "Select a GPU runtime in Colab."

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
).to("cuda").eval()

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Model device:", model.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

GPU: Tesla T4
PyTorch: 2.14.1+cu130
Transformers: 5.22.1
Model device: cuda:0


In [3]:
system_prompt = """Classify the student's message into exactly one intent.

schedule_interview: Book a new mock interview.
reschedule_interview: Move an existing mock interview to a different time or date.
cancel_interview: Cancel an existing mock interview.
check_schedule: Check the time, date, or booking status of an existing mock interview.
request_feedback: Ask for feedback on a completed mock interview.
out_of_scope: Ask for something outside the five mock interview actions above.

Return exactly one label as plain text."""

message = "Can we move my Friday mock interview to Monday?"

inputs = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": message},
    ],
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=24, do_sample=False)

prompt_length = inputs["input_ids"].shape[1]
new_tokens = output[0, prompt_length:]
prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Expected:  ", "reschedule_interview")
print("Prediction:", repr(prediction))
print("Exact match:", prediction == "reschedule_interview")

Expected:   reschedule_interview
Prediction: 'reschedule_interview'
Exact match: True


In [4]:
def predict_intent(text):
    inputs = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text},
        ],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=24, do_sample=False)

    prompt_length = inputs["input_ids"].shape[1]
    return tokenizer.decode(
        output[0, prompt_length:], skip_special_tokens=True
    ).strip()


pilot_examples = [
    ("I'd like to book my first mock interview next week.", "schedule_interview"),
    ("Can we move my Friday mock interview to Monday?", "reschedule_interview"),
    ("Please remove my Tuesday booking. I don't need another slot.", "cancel_interview"),
    ("What could I have done better in yesterday's mock?", "request_feedback"),
    ("What time is my upcoming mock interview scheduled for?", "check_schedule"),
    ("Can you help me rewrite the summary section of my resume?", "out_of_scope"),
]

correct = 0

for message, expected in pilot_examples:
    predicted = predict_intent(message)
    matched = predicted == expected
    correct += int(matched)
    print(f"{'PASS' if matched else 'MISS'} | {message}")
    print(f"  Expected: {expected} | Predicted: {predicted!r}")

print(f"\nPilot accuracy: {correct}/{len(pilot_examples)}")

MISS | I'd like to book my first mock interview next week.
  Expected: schedule_interview | Predicted: 'book_interview'
PASS | Can we move my Friday mock interview to Monday?
  Expected: reschedule_interview | Predicted: 'reschedule_interview'
PASS | Please remove my Tuesday booking. I don't need another slot.
  Expected: cancel_interview | Predicted: 'cancel_interview'
MISS | What could I have done better in yesterday's mock?
  Expected: request_feedback | Predicted: 'check_schedule'
PASS | What time is my upcoming mock interview scheduled for?
  Expected: check_schedule | Predicted: 'check_schedule'
MISS | Can you help me rewrite the summary section of my resume?
  Expected: out_of_scope | Predicted: 'Sure! Here’s a revised version:\n\n---\n\n**Summary:** Enhance your skills and showcase your potential by refining your resume'

Pilot accuracy: 3/6


importing the data and run the model

In [5]:
import json
from google.colab import files

uploaded = files.upload()
rows = [
    json.loads(line)
    for line in uploaded["tndrr_task1_intents.jsonl"].splitlines()
    if line.strip()
]

train_rows = [row for row in rows if row["split"] == "train"]
test_rows = [row for row in rows if row["split"] == "test"]

print(f"Training rows: {len(train_rows)} | Test rows: {len(test_rows)}")

base_predictions = []

for row in test_rows:
    prediction = predict_intent(row["text"])
    base_predictions.append({
        "text": row["text"],
        "expected": row["label"],
        "predicted": prediction,
        "correct": prediction == row["label"],
    })

correct = sum(row["correct"] for row in base_predictions)
print(f"Base accuracy: {correct}/{len(test_rows)} ({correct / len(test_rows):.1%})")

with open("base_predictions.jsonl", "w", encoding="utf-8") as output:
    for row in base_predictions:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

files.download("base_predictions.jsonl")

Saving tndrr_task1_intents.jsonl to tndrr_task1_intents.jsonl
Training rows: 120 | Test rows: 60
Base accuracy: 21/60 (35.0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Prepare the training examples


In [6]:
%pip install -q peft

def encode_training_row(row):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": row["text"]},
    ]

    prompt_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True
    )["input_ids"]

    full_ids = tokenizer.apply_chat_template(
        messages + [{"role": "assistant", "content": row["label"]}],
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
    )["input_ids"]

    prompt_length = len(prompt_ids)
    assert full_ids[:prompt_length] == prompt_ids, "Prompt tokens do not match."

    end_of_turn = tokenizer.convert_tokens_to_ids("<|im_end|>")
    answer_end = full_ids.index(end_of_turn, prompt_length) + 1
    input_ids = full_ids[:answer_end]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * prompt_length + input_ids[prompt_length:],
    }

training_examples = [encode_training_row(row) for row in train_rows]
supervised_tokens = [
    token for token in training_examples[0]["labels"] if token != -100
]

print("Training examples:", len(training_examples))
print("Expected label:", train_rows[0]["label"])
print("Supervised text:", repr(tokenizer.decode(supervised_tokens)))

Training examples: 120
Expected label: schedule_interview
Supervised text: 'schedule_interview<|im_end|>'


Train the LoRA adapter


In [7]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [8]:
import torch
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed

assert not isinstance(model, PeftModel), "Reload the original Qwen model before training again."
set_seed(42)

model = get_peft_model(
    model,
    LoraConfig(
        task_type="CAUSAL_LM",
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
    ),
    autocast_adapter_dtype=True,
)
model.print_trainable_parameters()
model.config.use_cache = False

trainer = Trainer(
    model=model,
    train_dataset=training_examples,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer, label_pad_token_id=-100
    ),
    args=TrainingArguments(
        output_dir="lora_run",
        per_device_train_batch_size=4,
        learning_rate=2e-4,
        num_train_epochs=3,
        fp16=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        seed=42,
    ),
)

trainer.train()

model.config.use_cache = True
model.eval()
model.save_pretrained("qwen_mock_interview_lora")

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


Step,Training Loss
10,0.515098
20,0.273029
30,0.209026
40,0.133929
50,0.139210
60,0.110094
70,0.020982
80,0.132535
90,0.058935


In [9]:
import json
from pathlib import Path

def read_jsonl(filename):
    return [
        json.loads(line)
        for line in Path(filename).read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

rows = read_jsonl("tndrr_task1_intents.jsonl")
train_rows = [row for row in rows if row["split"] == "train"]
test_rows = [row for row in rows if row["split"] == "test"]
base_predictions = read_jsonl("base_predictions.jsonl")

print(f"Train: {len(train_rows)} | Test: {len(test_rows)} | Baseline: {len(base_predictions)}")

Train: 120 | Test: 60 | Baseline: 60


In [10]:
import json
from google.colab import files
from peft import PeftModel

assert isinstance(model, PeftModel), "The LoRA adapter needs to be loaded on model."
assert [(row["text"], row["label"]) for row in test_rows] == [
    (row["text"], row["expected"]) for row in base_predictions
], "The test rows must match the saved baseline."

model.eval()
lora_predictions = []

for row in test_rows:
    predicted = predict_intent(row["text"])
    lora_predictions.append({
        "text": row["text"],
        "expected": row["label"],
        "predicted": predicted,
        "correct": predicted == row["label"],
    })

total = len(test_rows)
base_correct = sum(row["correct"] for row in base_predictions)
lora_correct = sum(row["correct"] for row in lora_predictions)

print(f"Base accuracy: {base_correct}/{total} ({base_correct / total:.1%})")
print(f"LoRA accuracy: {lora_correct}/{total} ({lora_correct / total:.1%})")
print(f"Change: {100 * (lora_correct - base_correct) / total:+.1f} percentage points")

filename = "lora_predictions.jsonl"
with open(filename, "w", encoding="utf-8") as output:
    for row in lora_predictions:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

files.download(filename)

Base accuracy: 21/60 (35.0%)
LoRA accuracy: 42/60 (70.0%)
Change: +35.0 percentage points


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import json
from google.colab import files
from peft import PeftModel

assert isinstance(model, PeftModel), "The LoRA adapter needs to be loaded on model."
assert [(row["text"], row["label"]) for row in test_rows] == [
    (row["text"], row["expected"]) for row in base_predictions
], "The test rows must match the saved baseline."

model.eval()
lora_predictions = []

for row in test_rows:
    predicted = predict_intent(row["text"])
    lora_predictions.append({
        "text": row["text"],
        "expected": row["label"],
        "predicted": predicted,
        "correct": predicted == row["label"],
    })

total = len(test_rows)
base_correct = sum(row["correct"] for row in base_predictions)
lora_correct = sum(row["correct"] for row in lora_predictions)

print(f"Base accuracy: {base_correct}/{total} ({base_correct / total:.1%})")
print(f"LoRA accuracy: {lora_correct}/{total} ({lora_correct / total:.1%})")
print(f"Change: {100 * (lora_correct - base_correct) / total:+.1f} percentage points")

filename = "lora_predictions.jsonl"
with open(filename, "w", encoding="utf-8") as output:
    for row in lora_predictions:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

files.download(filename)

Base accuracy: 21/60 (35.0%)
LoRA accuracy: 42/60 (70.0%)
Change: +35.0 percentage points


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>